[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LinkedInLearning/Fine-Tuning-for-LLMs-from-Beginner-to-Advanced/blob/main/CH05/05_03_end/notebooks/05_03_troubleshoot_training.ipynb)

# 05_03 Troubleshoot common training problems

Train a model badly on purpose, so you can recognize overfitting and catastrophic forgetting, and check how much GPU memory you're using.

Before you start, choose **Runtime > Change runtime type > T4 GPU**.

In [ ]:
%%capture
# Install Unsloth. This takes a minute or two.
!pip install unsloth

In [ ]:
# Pull the course files into this Colab session and load the Ridgeline helpers.
!git clone --quiet https://github.com/LinkedInLearning/Fine-Tuning-for-LLMs-from-Beginner-to-Advanced.git course 2>/dev/null || echo "Course files already downloaded."

import sys
sys.path.insert(0, "course/CH05")

import ridgeline

## Load the model, LoRA, and data

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen3-4B-Instruct-2507",
    max_seq_length=2048,
    load_in_4bit=True,
)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,                   # rank: how much room the adapter has to learn
    lora_alpha=16,          # scaling: how strongly the adapter's update counts
    lora_dropout=0,         # Unsloth is optimized for 0
    target_modules=[        # the layers that get adapters
        "q_proj", "k_proj", "v_proj", "o_proj",   # attention
        "gate_proj", "up_proj", "down_proj",      # feed-forward
    ],
    bias="none",
    use_gradient_checkpointing="unsloth",       # saves memory
    random_state=3407,
)
model.print_trainable_parameters()

In [ ]:
from datasets import load_dataset

# Use the course checkpoint, or upload your own files from Chapter 4 and change these paths.
TRAIN_FILE = "course/CH05/data/train.jsonl"
EVAL_FILE = "course/CH05/data/eval.jsonl"

dataset = load_dataset("json", data_files={"train": TRAIN_FILE, "eval": EVAL_FILE})


def to_text(example):
    """Render each conversation with the model's own chat template."""
    return {"text": ridgeline.chat_text(tokenizer, example["messages"])}


dataset = dataset.map(to_text, remove_columns=dataset["train"].column_names)
print(dataset)

In [ ]:
# Use a small slice of the data so the model overfits quickly.
small_train = dataset["train"].select(range(60))
small_eval = dataset["eval"].select(range(24))
print(f"{len(small_train)} training examples, {len(small_eval)} evaluation examples")

## Overtrain on purpose

In [ ]:
from trl import SFTConfig, SFTTrainer
from unsloth import is_bfloat16_supported
from unsloth.chat_templates import train_on_responses_only

# Too many epochs and too high a learning rate, on purpose.
config = SFTConfig(
    output_dir="outputs",
    dataset_text_field="text",
    max_length=2048,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=8,
    learning_rate=5e-4,
    warmup_steps=2,
    optim="adamw_8bit",
    logging_steps=2,
    eval_strategy="steps",
    eval_steps=4,
    per_device_eval_batch_size=4,
    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),
    seed=3407,
    report_to="none",
)

trainer = SFTTrainer(model=model, processing_class=tokenizer, train_dataset=small_train,
                     eval_dataset=small_eval, args=config)
trainer = train_on_responses_only(trainer, instruction_part="<|im_start|>user\n",
                                  response_part="<|im_start|>assistant\n")
trainer.evaluate()
trainer.train()

## Spot overfitting in the loss curves

Watch evaluation loss. It falls at first, then turns back up while training loss keeps dropping. That's the model memorizing its 60 examples.

In [ ]:
import matplotlib.pyplot as plt

history = trainer.state.log_history
train_points = [(h["step"], h["loss"]) for h in history if "loss" in h]
eval_points = [(h["step"], h["eval_loss"]) for h in history if "eval_loss" in h]

plt.plot(*zip(*train_points), label="training loss")
plt.plot(*zip(*eval_points), marker="o", label="evaluation loss")
plt.xlabel("step")
plt.ylabel("loss")
plt.legend()
plt.show()

## Check GPU memory

If a run crashes with an out-of-memory error, this number tells you how close you were to the limit.

In [ ]:
import torch

peak = torch.cuda.max_memory_reserved() / 1e9
total = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"Peak GPU memory: {peak:.1f} GB of {total:.1f} GB")

## Test for catastrophic forgetting

Ask an ordinary question that has nothing to do with support tickets. `disable_adapter` lets you compare against the original model without reloading it.

In [ ]:
FastLanguageModel.for_inference(model)
question = [{"role": "user", "content": "Give me three tips for my first overnight hike."}]

with model.disable_adapter():
    print("BASE MODEL\n" + ridgeline.generate(model, tokenizer, question, max_new_tokens=200))

print("\nOVERTRAINED MODEL\n" + ridgeline.generate(model, tokenizer, question, max_new_tokens=200))

## Quick fixes

| Symptom | What's happening | Try this |
| --- | --- | --- |
| `CUDA out of memory` | The batch doesn't fit on the GPU | Lower `per_device_train_batch_size` and raise `gradient_accumulation_steps` to match. Lower `max_length` or the rank. |
| Evaluation loss rises while training loss keeps falling | Overfitting: the model is memorizing | Fewer epochs, a lower learning rate, or more varied data |
| Loss jumps around or spikes | The learning rate is too high | Halve the learning rate |
| Loss barely moves | The learning rate is too low, or there's too little data | Raise the learning rate or add data |
| Ordinary questions get ticket-style answers | Catastrophic forgetting | Fewer epochs, a lower learning rate, or mix in some general examples |
| Replies are garbled or never stop | The chat template doesn't match | Format training data and prompts with the tokenizer's own template |